# 🧮 NOVA Tutorial 5: Integrated Math & Visualization ToolsNOVA includes integrated mathematical tools optimized for astronomical dataprocessing, and easy-to-use visualization functions. No external dependenciesbeyond NumPy and matplotlib are required.**Math Tools (`nova.math`):**- Sigma clipping & robust statistics- FFT-based 2D convolution & smoothing- Image rebinning & resizing- Image stacking (mean, median, sigma-clip)- Background estimation- Source detection- Aperture photometry- Continuum normalization (spectral)- Cosmic ray cleaning**Visualization Tools (`nova.viz`):**- `display_image` — Quick-look astronomical images- `display_rgb` — Three-colour composites- `display_spectrum` — 1-D spectral plots- `display_histogram` — Pixel distributions- `display_cutout` — Region zoom- `display_comparison` — Side-by-side comparison- `display_mosaic` — Multi-panel grids- `display_provenance` — Provenance chain diagrams

In [ ]:
# Setupimport syssys.path.insert(0, "../nova-py")import numpy as npimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltimport novafrom nova import math as nmathfrom nova import vizprint(f"NOVA version: {nova.__version__}")

## Part 1: Statistical Tools### 1.1 Sigma ClippingIteratively reject outliers from data — essential for astronomical background estimation.

In [ ]:
# Generate data with outliersrng = np.random.default_rng(42)data = rng.normal(100, 10, 10000)data[:50] = rng.uniform(500, 1000, 50)  # Outliers# Sigma clipclipped = nmath.sigma_clip(data, sigma=3.0)print(f"Original: {len(data)} points")print(f"After 3σ clip: {clipped.compressed().size} points ({clipped.mask.sum()} rejected)")# Compare statisticsprint(f"\nBefore clipping: mean={np.mean(data):.1f}, std={np.std(data):.1f}")print(f"After clipping:  mean={np.mean(clipped.compressed()):.1f}, std={np.std(clipped.compressed()):.1f}")

### 1.2 Robust StatisticsBiweight estimators are resistant to outliers — ideal for real astronomical data.

In [ ]:
# Robust statisticsstats = nmath.robust_statistics(data)print("Robust Statistics:")for key, val in stats.items():    print(f"  {key:>20s}: {val:.3f}")# Compare with sigma-clipped statssc_stats = nmath.sigma_clipped_stats(data, sigma=3.0)print("\nSigma-Clipped Stats:")for key, val in sc_stats.items():    print(f"  {key:>8s}: {val:.3f}" if isinstance(val, float) else f"  {key:>8s}: {val}")

## Part 2: Image Convolution & Smoothing### 2.1 Gaussian Kernels

In [ ]:
# Create Gaussian kernels of different sizesfig, axes = plt.subplots(1, 4, figsize=(16, 3))for ax, sigma in zip(axes, [0.5, 1.0, 2.0, 5.0]):    kernel = nmath.gaussian_kernel_2d(sigma)    ax.imshow(kernel, cmap="hot")    ax.set_title(f"σ = {sigma}, size = {kernel.shape[0]}", fontsize=10)    ax.axis("off")plt.suptitle("Gaussian Kernels", fontsize=14, fontweight="bold")plt.tight_layout()plt.show()

### 2.2 FFT Convolution with NaN Handling

In [ ]:
# Create a test image with sources and NaN pixelsimage = np.zeros((128, 128))yy, xx = np.ogrid[0:128, 0:128]image += 1000 * np.exp(-((xx-64)**2 + (yy-64)**2) / (2*5**2))image += 500 * np.exp(-((xx-30)**2 + (yy-90)**2) / (2*3**2))image += rng.normal(0, 5, (128, 128))# Add NaN holes (simulating bad pixels)image[50:55, 50:55] = np.nan# Smooth with NaN interpolationkernel = nmath.gaussian_kernel_2d(2.0)smoothed = nmath.convolve_fft(image, kernel, nan_treatment="interpolate")fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].imshow(image, origin="lower", cmap="gray")axes[0].set_title("Original (with NaN holes)")axes[1].imshow(smoothed, origin="lower", cmap="gray")axes[1].set_title("Smoothed (NaN interpolated)")axes[2].imshow(image - smoothed, origin="lower", cmap="RdBu_r")axes[2].set_title("Residual")plt.tight_layout()plt.show()

## Part 3: Image Rebinning & Resizing

In [ ]:
# Create a test imagerng = np.random.default_rng(42)full_image = rng.poisson(200, (256, 256)).astype(float)yy, xx = np.ogrid[0:256, 0:256]full_image += 5000 * np.exp(-((xx-128)**2 + (yy-128)**2) / (2*10**2))# Rebin (downsample by summing)rebinned = nmath.rebin(full_image, (64, 64), method="sum")# Resize (interpolated)resized_up = nmath.resize_image(rebinned, (256, 256), order=1)fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].imshow(full_image, origin="lower", cmap="gray")axes[0].set_title(f"Original ({full_image.shape})")axes[1].imshow(rebinned, origin="lower", cmap="gray")axes[1].set_title(f"Rebinned ({rebinned.shape})")axes[2].imshow(resized_up, origin="lower", cmap="gray")axes[2].set_title(f"Resized back ({resized_up.shape})")plt.tight_layout()plt.show()print(f"Original total flux: {full_image.sum():.0f}")print(f"Rebinned total flux: {rebinned.sum():.0f} (conserved!)")

## Part 4: Spectral Analysis Tools### 4.1 Continuum Normalization

In [ ]:
# Generate a realistic stellar spectrumwavelength = np.linspace(3800, 7200, 2048)# Blackbody-ish continuumT = 5800  # Solar temperatureflux = 1e10 / (wavelength**5 * (np.exp(1.44e7 / (wavelength * T)) - 1))flux = flux / np.max(flux) * 1000# Add absorption lineslines = [    (3934, 30, 0.7, "Ca II K"),    (3968, 25, 0.6, "Ca II H"),    (4861, 35, 0.7, "Hβ"),    (5183, 15, 0.3, "Mg I b"),    (5890, 30, 0.8, "Na D"),    (6563, 40, 0.8, "Hα"),]for wl, width, depth, _ in lines:    flux *= 1 - depth * np.exp(-0.5 * ((wavelength - wl) / width * 5)**2)flux = np.abs(flux) + rng.normal(0, 2, len(flux))# Continuum normalizenorm_flux, continuum = nmath.continuum_normalize(wavelength, flux)# Plotfig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)ax1.plot(wavelength, flux, "b-", linewidth=0.5, alpha=0.7, label="Observed")ax1.plot(wavelength, continuum, "r-", linewidth=2, label="Continuum fit")ax1.set_ylabel("Flux")ax1.set_title("Stellar Spectrum with Continuum Fit", fontweight="bold")ax1.legend()ax2.plot(wavelength, norm_flux, "b-", linewidth=0.5)ax2.axhline(1.0, color="red", linestyle="--", alpha=0.5)for wl, _, _, name in lines:    ax2.axvline(wl, color="orange", linestyle=":", alpha=0.5)    ax2.text(wl, 0.3, name, rotation=90, fontsize=7, color="orange")ax2.set_xlabel("Wavelength (Å)")ax2.set_ylabel("Normalized Flux")ax2.set_title("Continuum-Normalized Spectrum", fontweight="bold")ax2.set_ylim(0, 1.3)plt.tight_layout()plt.show()

### 4.2 Equivalent Width Measurement

In [ ]:
# Measure equivalent widths of absorption linesprint("Equivalent Width Measurements:")print(f"{'Line':>10s}  {'λ (Å)':>8s}  {'EW (Å)':>8s}")print("-" * 32)for wl, _, _, name in lines:    ew = nmath.equivalent_width(wavelength, norm_flux, wl, 100)    print(f"{name:>10s}  {wl:8.0f}  {ew:8.2f}")

## Part 5: Visualization Tools Gallery### 5.1 Display Functions

In [ ]:
# Generate a rich test imagesky = rng.poisson(200, (256, 256)).astype(float)yy, xx = np.ogrid[0:256, 0:256]sky += 10000 * np.exp(-((xx-128)**2 + (yy-128)**2) / (2*8**2))sky += 5000 * np.exp(-((xx-60)**2 + (yy-180)**2) / (2*4**2))sky += 3000 * np.exp(-((xx-200)**2 + (yy-50)**2) / (2*6**2))# Different stretchesfig, axes = plt.subplots(2, 3, figsize=(15, 10))stretches = ["linear", "log", "sqrt", "asinh", "power"]for ax, stretch in zip(axes.flat, stretches):    fig_inner = viz.display_image(sky, title=f"Stretch: {stretch}",                                    stretch=stretch, colorbar=False, figsize=(5, 5))    # Re-render inline    from nova.visualization import _apply_stretch, _percentile_interval    vmin, vmax = _percentile_interval(sky, 99.5)    display_data = np.clip(sky, vmin, vmax)    display_data = _apply_stretch(display_data, stretch=stretch)    ax.imshow(display_data, origin="lower", cmap="gray")    ax.set_title(f"Stretch: {stretch}", fontweight="bold")    plt.close(fig_inner)axes.flat[-1].set_visible(False)plt.suptitle("Image Stretching Comparison", fontsize=16, fontweight="bold", y=1.02)plt.tight_layout()plt.show()

### 5.2 RGB Composites

In [ ]:
# Create 3-band images (simulating B, V, R filters)blue = rng.poisson(150, (256, 256)).astype(float)green = rng.poisson(200, (256, 256)).astype(float)  red = rng.poisson(250, (256, 256)).astype(float)# Same sources, different relative brightnessfor band, color_factor in [(blue, [1.2, 0.8, 0.5]),                            (green, [1.0, 1.0, 0.8]),                            (red, [0.5, 1.2, 1.5])]:    band += (color_factor[0] * 5000 * np.exp(-((xx-128)**2 + (yy-128)**2) / (2*8**2))).astype(float)    band += (color_factor[1] * 3000 * np.exp(-((xx-60)**2 + (yy-180)**2) / (2*4**2))).astype(float)    band += (color_factor[2] * 2000 * np.exp(-((xx-200)**2 + (yy-50)**2) / (2*6**2))).astype(float)fig = viz.display_rgb(red, green, blue, title="RGB Composite (R, V, B)", stretch="asinh")plt.show()

### 5.3 Spectral Display

In [ ]:
# Display spectrum with line markersline_markers = [    {"wavelength": 3934, "name": "Ca II K"},    {"wavelength": 4861, "name": "Hβ"},    {"wavelength": 5890, "name": "Na D"},    {"wavelength": 6563, "name": "Hα"},]fig = viz.display_spectrum(    wavelength, flux,    title="Simulated Stellar Spectrum",    xlabel="Wavelength (Å)",    ylabel="Flux (erg/s/cm²/Å)",    line_markers=line_markers,)plt.show()

### 5.4 Image Comparison & Cutout

In [ ]:
# Compare original vs smoothedsmoothed_sky = nmath.smooth_gaussian(sky, sigma=3.0)fig = viz.display_comparison(    sky, smoothed_sky,    title1="Original", title2="Smoothed (σ=3px)",    stretch="asinh",)plt.show()# Cutout of a sourcefig = viz.display_cutout(sky, center=(128, 128), size=30,                          title="Central Source", stretch="log")plt.show()

### 5.5 Provenance Visualization

In [ ]:
# Visualize a provenance chainprov = {    "@type": "prov:Bundle",    "prov:entity": [        {"@id": "raw_frame", "nova:entity_type": "RawObservation"},        {"@id": "flat_field", "nova:entity_type": "CalibrationFrame"},        {"@id": "calibrated", "nova:entity_type": "CalibratedImage"},        {"@id": "stacked", "nova:entity_type": "StackedImage"},    ],    "prov:activity": [        {"@id": "flat_division"},        {"@id": "image_stacking"},    ],    "prov:agent": [        {"@id": "nova_pipeline", "nova:name": "NOVA Pipeline v1.0"},        {"@id": "observer", "nova:name": "Dr. Smith"},    ],}fig = viz.display_provenance(prov, title="Data Processing Provenance")plt.show()

## 🎯 Summary: NOVA Integrated Tools### Math Tools (`nova.math`)| Category | Functions | Use Case ||----------|----------|----------|| **Statistics** | `sigma_clip`, `sigma_clipped_stats`, `robust_statistics` | Background estimation, outlier rejection || **Convolution** | `gaussian_kernel_2d`, `convolve_fft`, `smooth_gaussian` | PSF matching, noise reduction || **Resampling** | `rebin`, `resize_image` | Resolution matching, mosaicking || **Stacking** | `stack_images` (mean/median/σ-clip) | Combining exposures || **Background** | `estimate_background` | Sky subtraction || **Detection** | `detect_sources`, `aperture_photometry` | Source cataloging || **Spectral** | `continuum_normalize`, `equivalent_width` | Spectral analysis || **Cleaning** | `cosmic_ray_clean` | Artifact removal |### Visualization Tools (`nova.viz`)| Function | Description ||----------|-------------|| `display_image` | Quick-look with stretch, colormap, WCS || `display_rgb` | Three-colour composite || `display_spectrum` | 1-D plot with line markers || `display_histogram` | Pixel distribution with stats || `display_cutout` | Region zoom with coordinates || `display_comparison` | Side-by-side + difference || `display_mosaic` | Multi-panel grid || `display_provenance` | Provenance chain diagram |**All tools are NumPy-native and optimized for performance!**